# 0 — rough

Smoke test: can the LM solve a PDDL-style blocksworld with **two independent goals** (disjoint block sets)? Just prompt and look at the output.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,7"

import torch
from src.models import ModelandTokenizer

In [2]:
# MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
# MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
MODEL_NAME = "Qwen/Qwen3.6-27B"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}")

Qwen/Qwen3.6-27B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

n_layer=64, n_embd=5120


## Prompt

Two independent blocksworld instances with **disjoint block names** so every action unambiguously belongs to one of the two goals. PDDL-style action vocabulary: `pick-up`, `put-down`, `stack`, `unstack`.

In [10]:
USER_PROMPT = '''You are a blocksworld planner. The world has two independent instances. The robot arm can hold at most one block at a time.

State definitions:
  on-table(x)      -- block x is on the table.
  on(x, y)         -- block x is on block y.
  arm-empty        -- the robot arm is empty.
  holding(x)       -- the robot arm is holding block x.
  clear(x)         -- block x has nothing on top of it.

Available actions:
  (pick-up ?x)     -- pick up block x from the table. Prerequisite: arm-empty, on-table(?x), clear(?x).
  (put-down ?x)    -- put down block x on the table. Prerequisite: holding(?x).
  (stack ?x ?y)    -- stack x onto y. Prerequisite: holding(?x), clear(?y).
  (unstack ?x ?y)  -- pick up x from on top of y. Prerequisite: arm-empty, on(?x, ?y), clear(?x).

Instance 1:
  Initial state: on-table(A), on-table(B), on(C, A), arm-empty.
  Goal:          on(A, B), on(B, C).

Instance 2:
  Initial state: on-table(X), on(Y, X), on(Z, Y), clear(Z), arm-empty.
  Goal:          on(Z, X), on-table(Y).

Produce a single plan that achieves BOTH goals. Output exactly one action per line, in this format: (verb arg1) or (verb arg1 arg2). No commentary, no numbering, no blank lines. End with the line: DONE.
'''

messages = [
    {"role": "user", "content": USER_PROMPT},
]
prompt = mt.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False,
    # enable_thinking=False
)
print(prompt)

<|im_start|>user
You are a blocksworld planner. The world has two independent instances. The robot arm can hold at most one block at a time.

State definitions:
  on-table(x)      -- block x is on the table.
  on(x, y)         -- block x is on block y.
  arm-empty        -- the robot arm is empty.
  holding(x)       -- the robot arm is holding block x.
  clear(x)         -- block x has nothing on top of it.

Available actions:
  (pick-up ?x)     -- pick up block x from the table. Prerequisite: arm-empty, on-table(?x), clear(?x).
  (put-down ?x)    -- put down block x on the table. Prerequisite: holding(?x).
  (stack ?x ?y)    -- stack x onto y. Prerequisite: holding(?x), clear(?y).
  (unstack ?x ?y)  -- pick up x from on top of y. Prerequisite: arm-empty, on(?x, ?y), clear(?x).

Instance 1:
  Initial state: on-table(A), on-table(B), on(C, A), arm-empty.
  Goal:          on(A, B), on(B, C).

Instance 2:
  Initial state: on-table(X), on(Y, X), on(Z, Y), clear(Z), arm-empty.
  Goal:      

In [11]:
inputs = mt.tokenizer(prompt, return_tensors="pt").to(mt.model.device)
with torch.no_grad():
    out = mt._model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=mt.tokenizer.eos_token_id,
    )
gen_ids = out[0, inputs["input_ids"].shape[1]:]
generation = mt.tokenizer.decode(gen_ids, skip_special_tokens=True)
print(generation)

The user wants a plan for a Blocksworld problem with two independent instances.
The robot arm can hold at most one block.
The actions are standard Blocksworld actions: `pick-up`, `put-down`, `stack`, `unstack`.

**Instance 1:**
Initial state:
- `on-table(A)`
- `on-table(B)`
- `on(C, A)`
- `arm-empty`
- Implicitly: `clear(C)` (since nothing is on C), `clear(B)` (since nothing is on B). `A` is not clear because `C` is on it.

Goal state:
- `on(A, B)`
- `on(B, C)`

Let's analyze Instance 1.
Current: C is on A. A is on table. B is on table.
Target: A on B, B on C.
This implies a stack C -> B -> A (bottom to top).
Currently: A (bottom), C (top). B is separate.
To get A on B, A must be clear. Currently C is on A. So C must be moved.
To get B on C, B must be clear (it is) and C must be clear (it is).
Wait, if B is on C, then C is the base.
So the final stack is C (bottom), B (middle), A (top).
Let's check the goal predicates: `on(A, B)` means A is on B. `on(B, C)` means B is on C.
Yes, stack 